In [1]:
import os
import json
from collections import Counter
import numpy as np

PROC_DIR = '../data/processed'
PRED_DIR = '../data/predictions'
REPORT_DIR = '../data/reports'
RARE_IDX = 0
K_VALUES = [1, 3, 5, 9]
THRESHOLDS = [0.5, 0.4, 0.3, 0.2, 0.1]
os.makedirs(REPORT_DIR, exist_ok=True)

X_padded = np.load(f'{PROC_DIR}/X_padded.npy').astype(np.float32)
Y = np.load(f'{PROC_DIR}/y_target.npy').astype(np.float32)
test_ex = np.load(f'{PROC_DIR}/splits.npz')['test']
users = test_ex[:, 0]
y_test = Y[users].copy()
y_test[:, RARE_IDX] = 0
report = {}

In [2]:
def load_scores(name):
    s = np.load(f'{PRED_DIR}/{name}').astype(np.float32)
    return s if s.shape[0] == len(users) else s[users]

SCORES = {
    'LSTM': load_scores('lstm_test_scores.npy'),
    'Most Popular': load_scores('most_popular_scores.npy'),
    'FP-Growth': load_scores('fpgrowth_scores.npy'),
}


def rank_topk(scores, k):
    s = scores.copy()
    s[:, RARE_IDX] = -np.inf
    return np.argsort(-s, axis=1, kind='stable')[:, :k]


def per_user_metrics(scores, y, k):
    idx = rank_topk(scores, k)
    hits = np.take_along_axis(y, idx, axis=1).sum(axis=1)
    n_true = y.sum(axis=1)
    valid = n_true > 0
    hr = (hits > 0).astype(np.float32)
    rec = hits / np.maximum(n_true, 1)
    return hr, rec, valid


print(f"{'K':>3} | {'Model':<13} | {'HitRatio':>8} | {'Recall':>8}")
report['overall'] = {}
for k in K_VALUES:
    report['overall'][str(k)] = {}
    for name, s in SCORES.items():
        hr, rec, valid = per_user_metrics(s, y_test, k)
        report['overall'][str(k)][name] = {
            'HitRatio': float(hr[valid].mean()),
            'Recall': float(rec[valid].mean()),
        }
        print(f"{k:>3} | {name:<13} | {hr[valid].mean():>8.4f} | {rec[valid].mean():>8.4f}")

  K | Model         | HitRatio |   Recall
  1 | LSTM          |   0.1368 |   0.0601
  1 | Most Popular  |   0.1533 |   0.0707
  1 | FP-Growth     |   0.1533 |   0.0707
  3 | LSTM          |   0.1998 |   0.0927
  3 | Most Popular  |   0.1998 |   0.0927
  3 | FP-Growth     |   0.2034 |   0.0930
  5 | LSTM          |   0.2267 |   0.1032
  5 | Most Popular  |   0.2347 |   0.1044
  5 | FP-Growth     |   0.2374 |   0.1051
  9 | LSTM          |   0.3054 |   0.1389
  9 | Most Popular  |   0.2796 |   0.1219
  9 | FP-Growth     |   0.2829 |   0.1234


In [3]:
hist_len = (X_padded[users].sum(axis=2) > 0).sum(axis=1)
bins = [(1, 2), (3, 5), (6, 9), (10, 10)]
report['by_history'] = {}
print(f"{'History':>8} | {'Users':>6} | {'K':>3} | {'LSTM':>8} | {'Most Pop':>8} | {'FP-Growth':>9}")
for lo, hi in bins:
    m = (hist_len >= lo) & (hist_len <= hi)
    label = f"{lo}-{hi}"
    report['by_history'][label] = {'users': int(m.sum())}
    for k in [3, 9]:
        vals = {}
        for name, s in SCORES.items():
            hr, _, valid = per_user_metrics(s[m], y_test[m], k)
            vals[name] = float(hr[valid].mean()) if valid.any() else float('nan')
        report['by_history'][label][str(k)] = vals
        print(f"{label:>8} | {int(m.sum()):>6} | {k:>3} | {vals['LSTM']:>8.4f} | "
              f"{vals['Most Popular']:>8.4f} | {vals['FP-Growth']:>9.4f}")

 History |  Users |   K |     LSTM | Most Pop | FP-Growth
     1-2 |   3850 |   3 |   0.1574 |   0.1574 |    0.1632
     1-2 |   3850 |   9 |   0.2540 |   0.2449 |    0.2504
     3-5 |   5699 |   3 |   0.1946 |   0.1946 |    0.1964
     3-5 |   5699 |   9 |   0.3003 |   0.2769 |    0.2789
     6-9 |   2601 |   3 |   0.2578 |   0.2578 |    0.2604
     6-9 |   2601 |   9 |   0.3636 |   0.3298 |    0.3320
   10-10 |   1924 |   3 |   0.2212 |   0.2212 |    0.2274
   10-10 |   1924 |   9 |   0.3444 |   0.2890 |    0.2933


In [4]:
lstm = SCORES['LSTM']
top1_lstm = rank_topk(lstm, 1)[:, 0]
top1_mp = rank_topk(SCORES['Most Popular'], 1)[:, 0]
rare_top1 = float((np.argmax(lstm, axis=1) == RARE_IDX).mean())
rare_mean = float(lstm[:, RARE_IDX].mean())
other_mean = float(np.delete(lstm, RARE_IDX, axis=1).mean())
print(f"Users whose raw top-1 is RARE_ITEM: {rare_top1:.2%}")
print(f"Mean LSTM score: RARE_ITEM={rare_mean:.4f}, other items={other_mean:.4f}")
report['rare'] = {'top1_share': rare_top1, 'mean_rare': rare_mean, 'mean_other': other_mean}
report['top1_concentration'] = {}
for name, top1 in [('LSTM', top1_lstm), ('Most Popular', top1_mp)]:
    counts = Counter(top1.tolist())
    print(f"{name}: distinct top-1 items = {len(counts)}")
    for item, c in counts.most_common(5):
        print(f"    item index {item}: {c / len(top1):.2%} of users")
    report['top1_concentration'][name] = {
        'distinct': len(counts),
        'top5': [[int(i), c / len(top1)] for i, c in counts.most_common(5)],
    }

Users whose raw top-1 is RARE_ITEM: 0.00%
Mean LSTM score: RARE_ITEM=0.0000, other items=0.0029
LSTM: distinct top-1 items = 2
    item index 285: 72.50% of users
    item index 970: 27.50% of users
Most Popular: distinct top-1 items = 1
    item index 285: 100.00% of users


In [7]:
s = lstm.copy()
s[:, RARE_IDX] = 0
print(f"Max LSTM score over all users and items: {s.max():.4f}")
print(f"Mean of each user's top score: {s.max(axis=1).mean():.4f}")
n_true = y_test.sum(axis=1)
valid = n_true > 0
user_max = s.max(axis=1, keepdims=True)
print(f"{'Frac':>5} | {'Avg pred':>8} | {'Precision':>9} | {'Recall':>7} | {'Users w/ pred':>13}")
for frac in [0.9, 0.7, 0.5, 0.3]:
    pred = (s >= frac * user_max).astype(np.float32)
    n_pred = pred.sum(axis=1)
    hits = (pred * y_test).sum(axis=1)
    has = (n_pred > 0) & valid
    prec = float((hits[has] / n_pred[has]).mean())
    rec = float((hits[valid] / n_true[valid]).mean())
    print(f"{frac:>5.2f} | {n_pred[valid].mean():>8.2f} | {prec:>9.4f} | {rec:>7.4f} | "
          f"{has.sum() / valid.sum():>13.2%}")

Max LSTM score over all users and items: 0.0504
Mean of each user's top score: 0.0452
 Frac | Avg pred | Precision |  Recall | Users w/ pred
 0.90 |     1.45 |    0.1352 |  0.0756 |       100.00%
 0.70 |     1.81 |    0.1121 |  0.0802 |       100.00%
 0.50 |     3.00 |    0.0737 |  0.0927 |       100.00%
 0.30 |     8.76 |    0.0389 |  0.1343 |       100.00%


In [6]:
with open(f'{REPORT_DIR}/error_analysis_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print(f"Saved {REPORT_DIR}/error_analysis_report.json")

Saved ../data/reports/error_analysis_report.json
